# LeetCode 3718 — Smallest Missing Multiple of K bằng Python

Link đề: https://leetcode.com/problems/smallest-missing-multiple-of-k/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def missingMultiple(self, nums: List[int], k: int) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm (`missingMultiple`) đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho mảng `nums` và số nguyên `k`, tìm **bội dương nhỏ nhất của `k` không xuất hiện** trong `nums`.
2. Biết dùng **tập hợp (`set`)** để kiểm tra "có mặt hay không" trong `O(1)` thay vì quét cả mảng `O(n)`.
3. Hiểu vì sao đáp án luôn tồn tại và **không vượt quá `k * (n + 1)`** (nguyên lý chuồng bồ câu), nên vòng lặp luôn dừng.
4. Viết 4 cách: kiểm tra bằng `in` trên list, dùng `set`, mảng đánh dấu kích thước cố định (do `nums[i] <= 100`), và sắp xếp + con trỏ; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: bắt đầu từ `0`, chỉ xét đến `max(nums)`, nhầm với "số nguyên dương nhỏ nhất bị thiếu", và đếm bội mà không kiểm tra liên tiếp.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Hãy trả về **bội dương nhỏ nhất của `k` bị thiếu** trong `nums`. Một bội của `k` là số nguyên dương chia hết cho `k` (`k, 2k, 3k, ...`).

| Ví dụ | nums | k | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[8,2,3,4,6]` | `2` | `10` | Các bội của 2 là `2, 4, 6, 8, 10, ...`; `2, 4, 6, 8` đều có trong `nums`, `10` thì không |
| 2 | `[1,4,7,10,15]` | `5` | `5` | Các bội của 5 là `5, 10, 15, 20, ...`; `5` không có trong `nums` |

Constraints: `1 <= nums.length <= 100`, `1 <= nums[i] <= 100`, `1 <= k <= 100`.

In [1]:
def multiples(k, count):
    return [k * m for m in range(1, count + 1)]

for nums, k in [([8, 2, 3, 4, 6], 2), ([1, 4, 7, 10, 15], 5)]:
    ms = multiples(k, len(nums) + 1)
    print("nums =", nums, "k =", k)
    print("   các bội:", ms, "| có trong nums:", [m in nums for m in ms])

nums = [8, 2, 3, 4, 6] k = 2
   các bội: [2, 4, 6, 8, 10, 12] | có trong nums: [True, True, True, True, False, False]
nums = [1, 4, 7, 10, 15] k = 5
   các bội: [5, 10, 15, 20, 25, 30] | có trong nums: [False, True, True, False, False, False]


## 3. Kiến thức nền

### 3.1. Bội của `k` và phép chia dư `%`

`x` là bội của `k` khi `x % k == 0`. Các bội dương **liên tiếp** là `k, 2k, 3k, ...`, tức `k * m` với `m = 1, 2, 3, ...`. Hai cách sinh ra chúng: nhân `k * m`, hoặc cộng dồn `x += k`.

In [2]:
k = 5
print([x for x in range(1, 31) if x % k == 0])
print([k * m for m in range(1, 7)])
x = 0
for _ in range(6):
    x += k
    print(x, end=" ")
print()

[5, 10, 15, 20, 25, 30]
[5, 10, 15, 20, 25, 30]
5 10 15 20 25 30 


### 3.2. `in` trên `list` và trên `set`

`x in lst` quét từng phần tử nên tốn `O(n)`. `set` là bảng băm: kiểm tra `x in s` trung bình `O(1)`. Đổi `nums` thành `set(nums)` mất `O(n)` một lần, sau đó mỗi lần hỏi rất rẻ.

In [3]:
nums = [8, 2, 3, 4, 6]
lst = nums
st = set(nums)
print(10 in lst, 10 in st, 4 in lst, 4 in st)
print(st)               # tập không có thứ tự, không trùng lặp
print(len(set([1, 1, 1, 2])))

False False True True
{2, 3, 4, 6, 8}
2


### 3.3. Vòng lặp `while True` và `return` sớm

Khi không biết trước cần lặp bao nhiêu lần ta dùng `while True` và thoát bằng `return` khi tìm thấy. Ở bài này ta **chắc chắn** thoát vì chỉ có hữu hạn bội nằm trong `nums`. Hãy luôn tự chứng minh vòng lặp dừng được.

In [4]:
def first_not_in(seq_set, step):
    x = step
    while True:
        if x not in seq_set:
            return x
        x += step

print(first_not_in({2, 4, 6, 8}, 2))        # 10
print(first_not_in({1, 4, 7}, 5))           # 5

10
5


### 3.4. Mảng đánh dấu (boolean array) và `sorted`

Khi giá trị có **cận nhỏ** (`<= 100`), có thể dùng mảng `present = [False] * 101` với `present[v] = True` thay cho `set`. Còn `sorted(lst)` trả danh sách sắp xếp tăng; `set` cộng với `sorted` cho các giá trị khác nhau theo thứ tự.

In [5]:
present = [False] * 101
for v in [8, 2, 3, 4, 6]:
    present[v] = True
print([i for i, b in enumerate(present) if b])
print(sorted(set([5, 1, 5, 3, 1])))

[2, 3, 4, 6, 8]
[1, 3, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy liệt kê các bội của `k` **theo thứ tự tăng dần**, và với mỗi bội hỏi: "nó có trong `nums` không?". Bội đầu tiên **không** có là đáp án — vì ta đi từ nhỏ lên nên chắc chắn là nhỏ nhất.

Tính tay ví dụ 1 (`k = 2`, `nums = [8, 2, 3, 4, 6]`):

```
2  có trong nums  -> đi tiếp
4  có             -> đi tiếp
6  có             -> đi tiếp
8  có             -> đi tiếp
10 không có       -> đáp án = 10
```

Ví dụ 2 (`k = 5`): bội đầu tiên là `5`, mà `5` không có trong `[1, 4, 7, 10, 15]` -> đáp án `5` ngay.

**Vì sao luôn có đáp án?** `nums` chỉ có `n` phần tử, trong khi có vô hạn bội của `k`. Trong `n + 1` bội đầu tiên (`k, 2k, ..., (n+1)k`) tối đa `n` cái có mặt, nên **ít nhất một** cái vắng mặt. Suy ra đáp án `<= k * (n + 1)`.

### Bảng tính tay

| nums | k | các bội đã kiểm tra | kết quả |
|---|---|---|---|
| `[8,2,3,4,6]` | 2 | 2, 4, 6, 8 (có), 10 (thiếu) | 10 |
| `[1,4,7,10,15]` | 5 | 5 (thiếu) | 5 |
| `[3,6,9]` | 3 | 3, 6, 9 (có), 12 (thiếu) | 12 |
| `[1,2,3]` | 1 | 1, 2, 3 (có), 4 (thiếu) | 4 |
| `[7]` | 3 | 3 (thiếu) | 3 |

In [6]:
def oracle(nums, k):
    x = k
    while x in nums:
        x += k
    return x

for nums, k in [([8, 2, 3, 4, 6], 2), ([1, 4, 7, 10, 15], 5), ([3, 6, 9], 3), ([1, 2, 3], 1), ([7], 3)]:
    print(nums, k, "->", oracle(nums, k), "| cận trên k*(n+1) =", k * (len(nums) + 1))

[8, 2, 3, 4, 6] 2 -> 10 | cận trên k*(n+1) = 12
[1, 4, 7, 10, 15] 5 -> 5 | cận trên k*(n+1) = 30
[3, 6, 9] 3 -> 12 | cận trên k*(n+1) = 12
[1, 2, 3] 1 -> 4 | cận trên k*(n+1) = 4
[7] 3 -> 3 | cận trên k*(n+1) = 6


## 5. Cách 1 — Brute Force / baseline

Đúng như trực giác: với `m = 1, 2, 3, ...` lấy `x = k * m`, kiểm tra `x in nums` **trực tiếp trên list**, trả về `x` đầu tiên không có mặt.

Nút thắt: mỗi lần `x in nums` quét cả list `O(n)`, mà có tới `n + 1` bội để thử, nên tổng `O(n^2)`. Với `n <= 100` không sao, nhưng ta thấy ngay chỗ cải thiện: thay list bằng `set`.

In [7]:
class SolutionBrute:
    def missingMultiple(self, nums, k):
        m = 1
        while True:
            x = k * m
            if x not in nums:              # quét cả list: O(n)
                return x
            m += 1

print(SolutionBrute().missingMultiple([8, 2, 3, 4, 6], 2), SolutionBrute().missingMultiple([1, 4, 7, 10, 15], 5))

10 5


**Độ phức tạp:** Time `O(n^2)` (tối đa `n + 1` bội, mỗi lần quét `O(n)`), Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** việc "có mặt hay không" nên là phép **tra cứu `O(1)`**. Chuyển `nums` thành `set` một lần rồi đi dọc các bội tăng dần.

**Quy tắc "nếu… thì…":**

- Nếu bội hiện tại `x` **có** trong tập -> `x += k` và thử tiếp.
- Nếu `x` **không** có -> đó là đáp án (vì mọi bội nhỏ hơn đều đã có mặt).
- Vì `nums[i] <= 100` nên mọi bội `> 100` chắc chắn thiếu -> có thể dùng **mảng đánh dấu** cỡ 101 và không cần băm.

**Invariant:** trước mỗi lần kiểm tra `x`, tất cả bội `k, 2k, ..., x - k` đều đã được xác nhận có mặt trong `nums`.

In [8]:
# Kiểm chứng cận trên: đáp án luôn <= k * (n + 1)
import random
random.seed(5)
for _ in range(2000):
    n = random.randint(1, 100)
    k = random.randint(1, 100)
    nums = [random.randint(1, 100) for _ in range(n)]
    ans = oracle(nums, k)
    assert ans % k == 0 and ans <= k * (n + 1)
print("đáp án luôn là bội của k và <= k*(n+1)")

đáp án luôn là bội của k và <= k*(n+1)


## 7. Các cách tiếp theo

### Cách 2 — Dùng `set`

Tạo `seen = set(nums)` rồi đi qua bội `k, 2k, ...` bằng `x += k`. Mỗi lần tra cứu `O(1)` nên tổng `O(n)`. Đây là lời giải chuẩn.

In [9]:
class SolutionSet:
    def missingMultiple(self, nums, k):
        seen = set(nums)
        x = k
        while x in seen:
            x += k
        return x

print(SolutionSet().missingMultiple([8, 2, 3, 4, 6], 2), SolutionSet().missingMultiple([1, 4, 7, 10, 15], 5))

10 5


**Độ phức tạp:** Time `O(n)` (xây `set` `O(n)` + tối đa `n + 1` lần tra cứu `O(1)`), Space `O(n)`.

### Cách 3 — Mảng đánh dấu kích thước cố định

Do `1 <= nums[i] <= 100`, dùng `present = [False] * 101`. Với mỗi bội `x`: nếu `x > 100` thì **chắc chắn thiếu**, nếu `present[x]` là `False` thì cũng thiếu. Không cần băm, không cần `set`, và bộ nhớ là hằng số.

In [10]:
class SolutionArray:
    def missingMultiple(self, nums, k):
        present = [False] * 101              # chỉ số 0..100, vì nums[i] <= 100
        for v in nums:
            present[v] = True
        x = k
        while x <= 100 and present[x]:
            x += k
        return x                             # x > 100 (chắc chắn thiếu) hoặc present[x] == False

print(SolutionArray().missingMultiple([8, 2, 3, 4, 6], 2), SolutionArray().missingMultiple([1, 4, 7, 10, 15], 5))

10 5


**Độ phức tạp:** Time `O(n + 100/k)` tức `O(n)`, Space `O(1)` (mảng 101 phần tử cố định).

### Cách 4 — Sắp xếp rồi so khớp bằng con trỏ

Lọc các giá trị là bội của `k`, loại trùng, sắp tăng. Giữ biến `expected = k`. Duyệt từng giá trị `v` theo thứ tự: nếu `v == expected` thì `expected += k`; nếu `v > expected` thì có "lỗ hổng" -> dừng. Cuối cùng trả `expected`. Cách này không cần `set` tra cứu mà chỉ cần thứ tự.

In [11]:
class SolutionSorted:
    def missingMultiple(self, nums, k):
        mult = sorted({v for v in nums if v % k == 0})
        expected = k
        for v in mult:
            if v == expected:
                expected += k
            elif v > expected:
                break
        return expected

print(SolutionSorted().missingMultiple([8, 2, 3, 4, 6], 2), SolutionSorted().missingMultiple([1, 4, 7, 10, 15], 5))

10 5


**Độ phức tạp:** Time `O(n log n)` (sắp xếp), Space `O(n)`. Chậm hơn `set` nhưng minh họa kỹ thuật "so khớp với giá trị kỳ vọng" dùng nhiều trong bài toán *missing number*.

## 8. TRACE từng bước

Trace Cách 2 với `nums = [8, 2, 3, 4, 6]`, `k = 2`, `seen = {2, 3, 4, 6, 8}`:

| bước | x | x in seen? | hành động |
|---|---|---|---|
| 1 | 2 | có | x = 4 |
| 2 | 4 | có | x = 6 |
| 3 | 6 | có | x = 8 |
| 4 | 8 | có | x = 10 |
| 5 | 10 | không | trả về 10 |

Trace Cách 4: `mult = [2, 4, 6, 8]`, `expected = 2`: `v=2` khớp -> `4`; `v=4` -> `6`; `v=6` -> `8`; `v=8` -> `10`; hết dãy, trả `10`.

In [12]:
def trace_set(nums, k):
    seen = set(nums)
    print(f"nums={nums} k={k} seen={sorted(seen)}")
    x, step = k, 0
    while True:
        step += 1
        has = x in seen
        print(f"  bước {step}: x={x:>4} có trong seen? {has}")
        if not has:
            print("  => đáp án", x)
            return x
        x += k

assert trace_set([8, 2, 3, 4, 6], 2) == 10
assert trace_set([1, 4, 7, 10, 15], 5) == 5

nums=[8, 2, 3, 4, 6] k=2 seen=[2, 3, 4, 6, 8]
  bước 1: x=   2 có trong seen? True
  bước 2: x=   4 có trong seen? True
  bước 3: x=   6 có trong seen? True
  bước 4: x=   8 có trong seen? True
  bước 5: x=  10 có trong seen? False
  => đáp án 10
nums=[1, 4, 7, 10, 15] k=5 seen=[1, 4, 7, 10, 15]
  bước 1: x=   5 có trong seen? False
  => đáp án 5


In [13]:
def trace_sorted(nums, k):
    mult = sorted({v for v in nums if v % k == 0})
    print(f"nums={nums} k={k} bội của k trong nums (đã sắp xếp) = {mult}")
    expected = k
    for v in mult:
        if v == expected:
            print(f"  v={v} == expected={expected} -> expected={expected + k}")
            expected += k
        elif v > expected:
            print(f"  v={v} > expected={expected} -> lỗ hổng, dừng")
            break
    print("  => đáp án", expected)
    return expected

assert trace_sorted([8, 2, 3, 4, 6], 2) == 10
assert trace_sorted([3, 6, 12], 3) == 9

nums=[8, 2, 3, 4, 6] k=2 bội của k trong nums (đã sắp xếp) = [2, 4, 6, 8]
  v=2 == expected=2 -> expected=4
  v=4 == expected=4 -> expected=6
  v=6 == expected=6 -> expected=8
  v=8 == expected=8 -> expected=10
  => đáp án 10
nums=[3, 6, 12] k=3 bội của k trong nums (đã sắp xếp) = [3, 6, 12]
  v=3 == expected=3 -> expected=6
  v=6 == expected=6 -> expected=9
  v=12 > expected=9 -> lỗ hổng, dừng
  => đáp án 9


## 9. Code nộp LeetCode

In [14]:
from typing import List, Optional

class Solution:
    def missingMultiple(self, nums: List[int], k: int) -> int:
        seen = set(nums)
        x = k
        while x in seen:
            x += k
        return x

## 10. Vì sao thuật toán đúng

- **Đúng thứ tự:** ta thử các bội theo thứ tự tăng `k, 2k, 3k, ...`, nên bội đầu tiên không có mặt chính là **nhỏ nhất** trong các bội thiếu.
- **Invariant:** mỗi khi bước vào kiểm tra `x`, mọi bội dương nhỏ hơn `x` đều thuộc `nums`; nếu `x` cũng thuộc thì invariant còn giữ cho `x + k`.
- **Luôn dừng:** `nums` có tối đa `n` giá trị phân biệt, nhưng có `n + 1` bội đầu tiên `k, ..., (n+1)k` đôi một khác nhau. Theo nguyên lý chuồng bồ câu, ít nhất một bội thiếu, nên vòng lặp dừng sau tối đa `n + 1` lần.
- **Cách mảng đánh dấu:** mọi bội `> 100` chắc chắn thiếu (vì `nums[i] <= 100`), nên dừng khi `x > 100` vẫn cho đúng đáp án.
- **Cách sắp xếp:** nếu `v > expected` thì `expected` không xuất hiện ở các giá trị sau (vì danh sách tăng dần và đã loại trùng), nên `expected` là bội thiếu nhỏ nhất.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| `in` trên list | `O(n^2)` | `O(1)` |
| `set` | `O(n)` | `O(n)` |
| Mảng đánh dấu 101 ô | `O(n)` | `O(1)` |
| Sắp xếp + so khớp | `O(n log n)` | `O(n)` |

`n = len(nums) <= 100`. Số bội cần thử tối đa `n + 1`, nên phần "đi dọc bội" luôn `O(n)`.

## 12. Bộ test

Ví dụ của đề, các biên (`k = 1`, `k = 100`, mảng 1 phần tử, mảng đủ các bội) và so sánh **tất cả cách** trên nhiều input hợp lệ.

In [15]:
impls = [SolutionBrute(), SolutionSet(), SolutionArray(), SolutionSorted(), Solution()]

cases = [
    (([8, 2, 3, 4, 6], 2), 10), (([1, 4, 7, 10, 15], 5), 5),
    (([3, 6, 9], 3), 12), (([1, 2, 3], 1), 4), (([7], 3), 3),
    (([100], 100), 200), (([1], 1), 2), (([2], 1), 1),
    (([i for i in range(1, 101)], 1), 101),
    (([i for i in range(2, 101, 2)], 2), 102),
    (([20, 40, 60, 80, 100], 20), 120), (([5, 5, 5, 5], 5), 10),
]
for (nums, k), exp in cases:
    assert 1 <= len(nums) <= 100 and all(1 <= v <= 100 for v in nums) and 1 <= k <= 100
    for impl in impls:
        got = impl.missingMultiple(list(nums), k)
        assert got == exp, (type(impl).__name__, nums, k, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 12 ca


In [16]:
import random
random.seed(3718)
for _ in range(1000):
    n = random.randint(1, 100)
    k = random.randint(1, 100)
    mode = random.random()
    if mode < 0.5:
        nums = [random.randint(1, 100) for _ in range(n)]
    else:
        # ép nhiều bội của k xuất hiện để đáp án lớn
        pool = [x for x in range(k, 101, k)] or [1]
        nums = [random.choice(pool) for _ in range(n)] if random.random() < 0.5 else (pool * 3)[:max(1, min(n, len(pool)))]
    nums = nums[:100]
    exp = oracle(nums, k)
    outs = [impl.missingMultiple(list(nums), k) for impl in impls]
    assert all(o == exp for o in outs), (nums, k, outs, exp)
    assert exp % k == 0 and exp <= k * (len(nums) + 1)
print("1000 test ngẫu nhiên (seed=3718): 5 cách cùng đáp án, đáp án là bội của k")

1000 test ngẫu nhiên (seed=3718): 5 cách cùng đáp án, đáp án là bội của k


In [17]:
# Các trường hợp đáp án lớn hơn 100 (chỉ xảy ra khi mọi bội <= 100 đều có mặt)
big = 0
for k in range(1, 101):
    nums = list(range(k, 101, k))
    exp = oracle(nums, k)
    assert all(impl.missingMultiple(list(nums), k) == exp for impl in impls)
    assert exp == (100 // k + 1) * k > 100
    big += 1
print("k = 1..100 với nums = mọi bội <= 100: đáp án luôn > 100;", big, "ca")

k = 1..100 với nums = mọi bội <= 100: đáp án luôn > 100; 100 ca


## 13. Các lỗi người mới hay gặp

### Lỗi 1: bắt đầu từ `0`

`0` chia hết cho mọi `k` nhưng **không phải số dương**; vì `nums[i] >= 1` nên `0` luôn "thiếu" và hàm trả `0` sai.

In [18]:
def wrong_start_zero(nums, k):
    seen = set(nums)
    x = 0                                  # SAI: phải bắt đầu từ k
    while x in seen:
        x += k
    return x

print(wrong_start_zero([8, 2, 3, 4, 6], 2), "!=", Solution().missingMultiple([8, 2, 3, 4, 6], 2))
assert wrong_start_zero([8, 2, 3, 4, 6], 2) == 0 != 10

0 != 10


### Lỗi 2: chỉ xét các bội không vượt `max(nums)`

Khi mọi bội `<= max(nums)` đều có mặt, đáp án lại nằm **ngoài** khoảng đó (ví dụ `10 > 8`). Dừng sớm tại `max(nums)` làm hàm không trả được gì.

In [19]:
def wrong_upto_max(nums, k):
    seen = set(nums)
    for x in range(k, max(nums) + 1, k):   # SAI: bỏ qua các bội lớn hơn max
        if x not in seen:
            return x
    return -1

print(wrong_upto_max([8, 2, 3, 4, 6], 2), "!=", Solution().missingMultiple([8, 2, 3, 4, 6], 2))
assert wrong_upto_max([8, 2, 3, 4, 6], 2) == -1 and Solution().missingMultiple([8, 2, 3, 4, 6], 2) == 10

-1 != 10


### Lỗi 3: nhầm với "số nguyên dương nhỏ nhất bị thiếu"

Đề chỉ xét các **bội của `k`**, không phải mọi số nguyên dương.

In [20]:
def wrong_mex(nums, k):
    seen = set(nums)
    x = 1
    while x in seen:                       # SAI: bước nhảy 1 thay vì k
        x += 1
    return x

print(wrong_mex([1, 2, 3, 4], 2), "!=", Solution().missingMultiple([1, 2, 3, 4], 2))
assert wrong_mex([1, 2, 3, 4], 2) == 5 and Solution().missingMultiple([1, 2, 3, 4], 2) == 6

5 != 6


### Lỗi 4: đếm số bội có mặt rồi suy ra đáp án

Số bội có mặt **không** cho biết bội nào thiếu: các bội có mặt chưa chắc liên tiếp từ `k`.

In [21]:
def wrong_count(nums, k):
    cnt = len({v for v in nums if v % k == 0})
    return (cnt + 1) * k                   # SAI: giả sử các bội có mặt là k, 2k, ..., cnt*k

print(wrong_count([4], 2), "!=", Solution().missingMultiple([4], 2))      # 4 vs 2
assert wrong_count([4], 2) == 4 and Solution().missingMultiple([4], 2) == 2

4 != 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số nhỏ nhất **không có** trong mảng, với điều kiện X" -> đưa mảng vào `set`, rồi **duyệt ứng viên theo thứ tự tăng** đến khi gặp ứng viên không thuộc tập.
- Cận nhỏ của giá trị (`<= 100`) gợi ý **mảng đánh dấu**.
- Luôn tự hỏi "vòng lặp có dừng không?" -> nguyên lý chuồng bồ câu: `n` phần tử chỉ "chiếm" tối đa `n` ứng viên.
- Dùng `x += k` thay vì `k * m` để tránh biến `m` thừa.
- Tránh nhầm "bội dương" (bắt đầu từ `k`) với "bội không âm" (bắt đầu từ `0`).
- Tự kiểm tra nhanh: `k = 1`, `nums = [1..n]` thì đáp án là `n + 1`.

## 15. Biến thể của bài

1. **First Missing Positive** (LeetCode 41): số nguyên dương nhỏ nhất bị thiếu, yêu cầu `O(n)` thời gian và `O(1)` bộ nhớ phụ.
2. **Missing Number** (268): thiếu một số trong `0..n`; dùng tổng hoặc XOR.
3. **Smallest Missing Non-negative Integer After Operations** (2598): cộng/trừ bội của `value` rồi tìm MEX; đếm theo dư.
4. **Find Missing and Repeated Values** (2965).
5. **Bội nhỏ nhất lớn hơn `low` và không nằm trong tập cấm** (chính là bài tập 5 bên dưới).
6. Phiên bản có `nums[i]` rất lớn (`10^9`): bắt buộc dùng `set`, không dùng mảng đánh dấu.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| `in` trên list | quét list mỗi lần | `O(n^2)` | `O(1)` | Viết oracle / bài siêu nhỏ |
| `set` | tra cứu băm `O(1)` | `O(n)` | `O(n)` | Nộp bài, tổng quát với giá trị lớn |
| Mảng đánh dấu | `present[v]` cỡ 101 | `O(n)` | `O(1)` | Giá trị có cận nhỏ |
| Sắp xếp + so khớp | `expected` tăng dần | `O(n log n)` | `O(n)` | Khi không muốn băm, hoặc dữ liệu đã sắp xếp |

Nộp cách `set` (ngắn nhất, đúng với mọi cận).

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().missingMultiple([6, 3, 9, 12], 3)` ra gì?

**Bài 2 (tính tay):** với `nums = [10, 20, 40, 50]`, `k = 10`, liệt kê các bội đã kiểm tra và đáp án.

**Bài 3 (điền TODO):** hoàn thành `missing_multiple_count(nums, k)` trả về **số bội** của `k` (không vượt quá `100`) **không** xuất hiện trong `nums`.

**Bài 4 (tự viết lại):** viết `missing_multiple_next(nums, k)` bằng `next(...)` và `itertools.count(k, k)`, không dùng vòng `while`.

**Bài 5 (mini challenge):** viết `smallest_missing_multiple_above(nums, k, low)` trả về bội dương nhỏ nhất của `k` **lớn hơn `low`** và không có trong `nums`.

In [22]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().missingMultiple([6, 3, 9, 12], 3))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def missing_multiple_count_todo(nums, k):
    # TODO: đếm các bội k, 2k, ... <= 100 không nằm trong set(nums)
    return 0

15


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: các bội của 3 là `3, 6, 9, 12, 15, ...`. Bài 2: bội của 10 là `10, 20, 30, ...`.

**Mức 2 (vừa):** Bài 3: dùng `seen = set(nums)` và `sum(1 for x in range(k, 101, k) if x not in seen)`. Bài 4: `next(x for x in count(k, k) if x not in seen)`. Bài 5: bắt đầu từ bội đầu tiên lớn hơn `low`: `(low // k + 1) * k`.

**Mức 3 (gần đáp án):** Bài 1: `15`. Bài 2: `10, 20` có, `30` thiếu -> `30`. Bài 5: `x = (low // k + 1) * k` rồi `while x in seen: x += k`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
from itertools import count

# Bài 1, 2
assert Solution().missingMultiple([6, 3, 9, 12], 3) == 15
assert Solution().missingMultiple([10, 20, 40, 50], 10) == 30

# Bài 3
def missing_multiple_count(nums, k):
    seen = set(nums)
    return sum(1 for x in range(k, 101, k) if x not in seen)
assert missing_multiple_count([8, 2, 3, 4, 6], 2) == 50 - 4          # 50 bội <= 100, 4 có mặt
assert missing_multiple_count([1, 4, 7, 10, 15], 5) == 20 - 2        # 20 bội <= 100; có mặt 10 và 15
assert missing_multiple_count(list(range(1, 101)), 7) == 0

# Bài 4
def missing_multiple_next(nums, k):
    seen = set(nums)
    return next(x for x in count(k, k) if x not in seen)
for nums, k in [([8, 2, 3, 4, 6], 2), ([1, 4, 7, 10, 15], 5), ([6, 3, 9, 12], 3)]:
    assert missing_multiple_next(nums, k) == Solution().missingMultiple(nums, k)

# Bài 5
def smallest_missing_multiple_above(nums, k, low):
    seen = set(nums)
    x = (low // k + 1) * k
    while x in seen:
        x += k
    return x
assert smallest_missing_multiple_above([8, 2, 3, 4, 6], 2, 0) == 10
assert smallest_missing_multiple_above([8, 2, 3, 4, 6], 2, 4) == 10
assert smallest_missing_multiple_above([6, 3, 9, 12], 3, 4) == 15
assert smallest_missing_multiple_above([], 5, 12) == 15
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "ứng viên nhỏ nhất không thuộc tập":**

```python
seen = set(nums)
x = k                      # ứng viên đầu tiên (bội dương nhỏ nhất)
while x in seen:
    x += k                 # sang ứng viên kế tiếp
return x
```

**Checklist trước khi submit:**

- [ ] Bắt đầu từ `k` (không phải `0`)?
- [ ] Không giới hạn nhầm theo `max(nums)`; vòng lặp tự dừng vì nguyên lý chuồng bồ câu?
- [ ] Dùng `set` thay vì `in` trên list để tra cứu `O(1)`?
- [ ] Thử `k = 1`, `k = 100`, mảng đầy đủ các bội (đáp án vượt `100`), mảng 1 phần tử?
- [ ] Chữ ký đúng `missingMultiple(self, nums: List[int], k: int) -> int`?
- [ ] Đối chiếu với brute-force trên input ngẫu nhiên trong constraints?